# I13a — Genetic Algorithms and Evolutionary Computing

**COMPSCI 713 — Week 6: Learning & Adaptation**

---

## Overview

Genetic algorithms (GAs) are search and optimisation methods inspired by biological evolution. Instead of gradient descent, they evolve a **population** of candidate solutions through selection, crossover, and mutation.

In this lesson you will:
- Understand the genetic algorithm framework
- Implement selection, crossover, and mutation operators
- Solve optimisation problems with GAs
- Understand NEAT (NeuroEvolution of Augmenting Topologies)
- Compare evolutionary vs gradient-based optimisation

### COMPSCI 713 Alignment
- **Week 6 Monday:** NEAT (Genetic algorithms, neuro-genetic control)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from copy import deepcopy

## 1. The Genetic Algorithm Framework

```
1. Initialise random population
2. Evaluate fitness of each individual
3. REPEAT until termination:
   a. Selection  — choose parents based on fitness
   b. Crossover  — combine parents to create offspring
   c. Mutation   — randomly alter offspring
   d. Evaluate   — compute fitness of new individuals
   e. Replace    — form new population
4. Return best individual
```

**Key analogy to biology:**
- Individual = candidate solution
- Chromosome = encoding of the solution (e.g., binary string)
- Gene = one element of the encoding
- Fitness = how good the solution is
- Generation = one iteration of the loop

In [ ]:
# ── Genetic Algorithm: Maximise a function ─────────────────────────────────

class GeneticAlgorithm:
    def __init__(self, fitness_fn, gene_length, pop_size=50,
                 crossover_rate=0.8, mutation_rate=0.01, elitism=2):
        self.fitness_fn      = fitness_fn
        self.gene_length     = gene_length
        self.pop_size        = pop_size
        self.crossover_rate  = crossover_rate
        self.mutation_rate   = mutation_rate
        self.elitism         = elitism

    def _init_population(self):
        return [np.random.randint(0, 2, self.gene_length).tolist()
                for _ in range(self.pop_size)]

    def _fitness(self, individual):
        return self.fitness_fn(individual)

    def _tournament_select(self, population, fitnesses, k=3):
        """Tournament selection: pick k random individuals, return the best."""
        candidates = random.sample(range(len(population)), k)
        return population[max(candidates, key=lambda i: fitnesses[i])]

    def _single_point_crossover(self, parent1, parent2):
        if random.random() > self.crossover_rate:
            return parent1[:], parent2[:]
        point = random.randint(1, self.gene_length - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        return child1, child2

    def _mutate(self, individual):
        return [1 - gene if random.random() < self.mutation_rate else gene
                for gene in individual]

    def run(self, n_generations=100, verbose=True):
        population = self._init_population()
        history = {'best': [], 'avg': []}

        for gen in range(n_generations):
            fitnesses = [self._fitness(ind) for ind in population]

            best_fit = max(fitnesses)
            avg_fit  = sum(fitnesses) / len(fitnesses)
            history['best'].append(best_fit)
            history['avg'].append(avg_fit)

            if verbose and gen % 20 == 0:
                best_ind = population[fitnesses.index(best_fit)]
                print(f"Gen {gen:3d}: best={best_fit:.4f}, avg={avg_fit:.4f}")

            # Elitism: keep top individuals
            sorted_pop = [x for _, x in sorted(zip(fitnesses, population),
                          key=lambda p: -p[0])]
            new_pop = sorted_pop[:self.elitism]

            # Fill rest with crossover + mutation
            while len(new_pop) < self.pop_size:
                p1 = self._tournament_select(population, fitnesses)
                p2 = self._tournament_select(population, fitnesses)
                c1, c2 = self._single_point_crossover(p1, p2)
                new_pop.extend([self._mutate(c1), self._mutate(c2)])

            population = new_pop[:self.pop_size]

        fitnesses = [self._fitness(ind) for ind in population]
        best_idx  = fitnesses.index(max(fitnesses))
        return population[best_idx], max(fitnesses), history


# ── Problem: Maximise number of 1s (OneMax) ────────────────────────────────
def onemax(individual):
    return sum(individual) / len(individual)

ga = GeneticAlgorithm(onemax, gene_length=50, pop_size=100, mutation_rate=0.02)
best, best_fit, history = ga.run(n_generations=100)

plt.figure(figsize=(9, 4))
plt.plot(history['best'], label='Best fitness', color='green')
plt.plot(history['avg'],  label='Avg fitness',  color='blue', alpha=0.6)
plt.xlabel('Generation')
plt.ylabel('Fitness')
plt.title('Genetic Algorithm: OneMax Problem')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\nFinal best fitness: {best_fit:.4f} (target: 1.0)")

## 2. Real-Valued GA — Function Optimisation

In [ ]:
# ── Optimise a multimodal function ─────────────────────────────────────────
# f(x) = sin(x) + sin(10x/3) — has many local optima

def rastrigin(x_list):
    """Rastrigin function — classic multimodal benchmark. Minimum at x=0."""
    x = np.array(x_list)
    n = len(x)
    val = 10*n + np.sum(x**2 - 10*np.cos(2*np.pi*x))
    return -val  # negate because GA maximises

class RealValuedGA:
    def __init__(self, fitness_fn, n_dims, bounds=(-5.12, 5.12),
                 pop_size=100, mutation_std=0.1, crossover_rate=0.8, elitism=2):
        self.fitness_fn    = fitness_fn
        self.n_dims        = n_dims
        self.bounds        = bounds
        self.pop_size      = pop_size
        self.mutation_std  = mutation_std
        self.crossover_rate = crossover_rate
        self.elitism       = elitism

    def _init_pop(self):
        lo, hi = self.bounds
        return [np.random.uniform(lo, hi, self.n_dims).tolist()
                for _ in range(self.pop_size)]

    def _crossover(self, p1, p2):
        if random.random() > self.crossover_rate:
            return p1[:], p2[:]
        # Blend crossover (BLX-α)
        alpha = 0.5
        c1, c2 = [], []
        for g1, g2 in zip(p1, p2):
            lo = min(g1, g2) - alpha * abs(g1 - g2)
            hi = max(g1, g2) + alpha * abs(g1 - g2)
            c1.append(np.clip(random.uniform(lo, hi), *self.bounds))
            c2.append(np.clip(random.uniform(lo, hi), *self.bounds))
        return c1, c2

    def _mutate(self, ind):
        lo, hi = self.bounds
        return [np.clip(g + np.random.normal(0, self.mutation_std), lo, hi)
                for g in ind]

    def run(self, n_generations=200):
        pop = self._init_pop()
        history = []

        for _ in range(n_generations):
            fits = [self.fitness_fn(ind) for ind in pop]
            history.append(max(fits))

            sorted_pop = [x for _, x in sorted(zip(fits, pop), key=lambda p: -p[0])]
            new_pop = sorted_pop[:self.elitism]

            while len(new_pop) < self.pop_size:
                i1, i2 = random.sample(range(len(pop)), 2)
                p1 = pop[i1] if fits[i1] > fits[i2] else pop[i2]
                i3, i4 = random.sample(range(len(pop)), 2)
                p2 = pop[i3] if fits[i3] > fits[i4] else pop[i4]
                c1, c2 = self._crossover(p1, p2)
                new_pop.extend([self._mutate(c1), self._mutate(c2)])

            pop = new_pop[:self.pop_size]

        fits = [self.fitness_fn(ind) for ind in pop]
        best_idx = fits.index(max(fits))
        return pop[best_idx], max(fits), history


ga2 = RealValuedGA(rastrigin, n_dims=2, pop_size=200, mutation_std=0.3)
best2, best_fit2, hist2 = ga2.run(n_generations=300)

print(f"Best solution: x={[f'{v:.4f}' for v in best2]}")
print(f"Best fitness (negated Rastrigin): {best_fit2:.4f} (optimal: 0.0)")

plt.figure(figsize=(9, 4))
plt.plot(hist2, color='green')
plt.xlabel('Generation')
plt.ylabel('Best fitness (negated)')
plt.title('GA on Rastrigin Function (2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. NEAT — NeuroEvolution of Augmenting Topologies

NEAT (Stanley & Miikkulainen, 2002) evolves **both the weights and the structure** of neural networks.

Key innovations:
1. **Historical markings:** track gene origins to enable meaningful crossover between different topologies
2. **Speciation:** protect innovation by grouping similar networks — new structures get time to optimise before competing
3. **Complexification:** start simple, add nodes/connections incrementally

NEAT was used for:
- Learning to play Atari games
- Robot locomotion control
- Neuro-genetic control systems (COMPSCI 713 Week 6)

In [ ]:
# ── Simplified NEAT-inspired network evolution ─────────────────────────────
# We evolve weights of a fixed-topology network (simplified NEAT)

import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class SimpleNN:
    """A simple 2-layer neural network with evolvable weights."""
    def __init__(self, weights=None, n_in=2, n_hidden=4, n_out=1):
        self.n_in, self.n_hidden, self.n_out = n_in, n_hidden, n_out
        n_weights = n_in * n_hidden + n_hidden + n_hidden * n_out + n_out
        if weights is None:
            self.weights = np.random.randn(n_weights) * 0.5
        else:
            self.weights = np.array(weights)

    def forward(self, x):
        idx = 0
        W1 = self.weights[idx:idx+self.n_in*self.n_hidden].reshape(self.n_in, self.n_hidden)
        idx += self.n_in * self.n_hidden
        b1 = self.weights[idx:idx+self.n_hidden]
        idx += self.n_hidden
        W2 = self.weights[idx:idx+self.n_hidden*self.n_out].reshape(self.n_hidden, self.n_out)
        idx += self.n_hidden * self.n_out
        b2 = self.weights[idx:idx+self.n_out]
        h = sigmoid(x @ W1 + b1)
        return sigmoid(h @ W2 + b2)

    def n_weights_total(self):
        return len(self.weights)


# Evolve a network to learn XOR
XOR_X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
XOR_Y = np.array([[0],[1],[1],[0]], dtype=float)

def xor_fitness(weights):
    nn = SimpleNN(weights)
    preds = nn.forward(XOR_X)
    mse = np.mean((preds - XOR_Y) ** 2)
    return -mse  # maximise negative MSE

n_weights = SimpleNN().n_weights_total()

# Evolution strategy (simplified NEAT)
pop_size = 200
population = [np.random.randn(n_weights) * 0.5 for _ in range(pop_size)]
history = []

for gen in range(500):
    fits = [xor_fitness(ind) for ind in population]
    history.append(max(fits))

    # Sort by fitness
    sorted_pop = [x for _, x in sorted(zip(fits, population), key=lambda p: -p[0])]

    # Keep top 20%, generate rest via mutation + crossover
    n_elite = pop_size // 5
    new_pop = sorted_pop[:n_elite]

    while len(new_pop) < pop_size:
        parent = random.choice(sorted_pop[:n_elite])
        child = parent + np.random.randn(n_weights) * 0.1  # Gaussian mutation
        new_pop.append(child)

    population = new_pop

fits = [xor_fitness(ind) for ind in population]
best_weights = population[fits.index(max(fits))]
best_nn = SimpleNN(best_weights)

print("=== Evolved XOR Network ===")
print(f"{'Input':>10} {'Target':>8} {'Output':>8} {'Correct':>8}")
for x, y in zip(XOR_X, XOR_Y):
    pred = best_nn.forward(x.reshape(1,-1))[0,0]
    correct = '✓' if abs(pred - y[0]) < 0.5 else '✗'
    print(f"{str(x.astype(int)):>10} {int(y[0]):>8} {pred:>8.3f} {correct:>8}")

plt.figure(figsize=(9, 4))
plt.plot(history, color='purple')
plt.xlabel('Generation')
plt.ylabel('Best fitness (neg MSE)')
plt.title('NEAT-style Evolution: Learning XOR')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. GA vs Gradient Descent

| Aspect | Genetic Algorithm | Gradient Descent |
|---|---|---|
| **Requires gradient?** | ❌ No | ✅ Yes |
| **Handles non-differentiable?** | ✅ Yes | ❌ No |
| **Multimodal landscapes?** | ✅ Better | ❌ Gets stuck in local optima |
| **Scalability** | ❌ Slow for high-dim | ✅ Scales well |
| **Parallelisable?** | ✅ Naturally | ✅ With tricks |
| **Best for** | Discrete, combinatorial, black-box | Continuous, differentiable (neural nets) |

## 5. Summary

### Key Takeaways
- GAs evolve a **population** of solutions through selection, crossover, and mutation
- **Tournament selection** balances exploration and exploitation
- **Elitism** preserves the best solutions across generations
- **NEAT** evolves both weights and topology of neural networks
- GAs excel at problems where gradients are unavailable or the landscape is multimodal

### Next Steps
- **E07** — Meta-Learning (learning to learn, related to evolutionary strategies)
- **E10** — Deep Reinforcement Learning (another gradient-free approach: policy search)